# Physics-Informed Neural Network (IEEE 14-Bus)
This notebook setup up the envirnment, power flow model, and PINN. Training data created separately is used to train the PINN.

## Download dependencies for environment

In [ ]:
import sys
import subprocess
from pathlib import Path
from importlib.util import find_spec

def run_cmd(cmd):
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)

run_cmd([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])

requirements_file = Path("requirements.txt")
if requirements_file.exists():
    run_cmd([sys.executable, "-m", "pip", "install", "-r", str(requirements_file)])
else:
    fallback_packages = [
        "torch>=2.2",
        "numpy>=1.26",
        "pandas>=2.2",
        "matplotlib>=3.8",
        "scipy>=1.11",
        "networkx>=3.2",
        "pandapower>=3.1",
        "jupyter>=1.0",
        "ipykernel>=6.29",
    ]
    run_cmd([sys.executable, "-m", "pip", "install", *fallback_packages])

required_modules = ["torch", "numpy", "pandas", "matplotlib", "scipy", "networkx", "pandapower"]
missing_modules = [name for name in required_modules if find_spec(name) is None]

if missing_modules:
    raise ModuleNotFoundError(
        f"Setup completed, but these modules are still missing: {missing_modules}"
    )

import torch
print("Environment ready.")
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 1) Initialization

In [ ]:
import importlib
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Use IEEE 14-bus network

import pandapower as pp
import pandapower.networks as pn

net = pn.case14() # Load your project's 14-bus system

pp.runpp(net, numba=False)     # Run the power flow with Numba disabled

print("Model loaded and power flow executed successfully.")

## IEEE 14 Bus System
Single line diagram of system:

<img src="images/IEEE_14_bus_System.png" alt="IEEE 14 Bus System Diagram" width="400"/>

## Grid State: voltage magnitude and the phase angle
Role in PINN: Labels for the **Data Loss** term.

In [ ]:
print("=== STATE: net.res_bus ===")
state_cols = [c for c in ["vm_pu", "va_degree", "p_mw", "q_mvar"] if c in net.res_bus.columns]
print(net.res_bus[state_cols])

## Line Flows
Role in PINN: Measurements the attacker manipulates.

In [ ]:
print("=== FLOWS: net.res_line ===")
flow_cols = [c for c in ["p_from_mw", "q_from_mvar", "p_to_mw", "q_to_mvar", "loading_percent"] if c in net.res_line.columns]
print(net.res_line[flow_cols])

## Balance Generation and System Power Transfer
Role in PINN: Used to verify Kirchhoff’s Current Law (KCL).

In [ ]:
print("=== BALANCE: net.res_gen ===")
if len(net.res_gen) > 0:
    print(net.res_gen[[c for c in ["p_mw", "q_mvar"] if c in net.res_gen.columns]])
else:
    print("No generator results available.")

print("\n=== BALANCE: net.res_ext_grid ===")
print(net.res_ext_grid[[c for c in ["p_mw", "q_mvar"] if c in net.res_ext_grid.columns]])

## Topology: Line Connections and Transformers
Role in PINN: Physical network constraints used by the **Physics Loss** term.

In [ ]:
print("=== TOPOLOGY: net.line ===")
line_cols = [c for c in ["from_bus", "to_bus", "length_km", "r_ohm_per_km", "x_ohm_per_km", "c_nf_per_km"] if c in net.line.columns]
print(net.line[line_cols])

print("\n=== TOPOLOGY: net.trafo ===")
if len(net.trafo) > 0:
    trafo_cols = [c for c in ["hv_bus", "lv_bus", "sn_mva", "vk_percent", "vkr_percent"] if c in net.trafo.columns]
    print(net.trafo[trafo_cols])
else:
    print("No transformers in this network model.")

## Extract data from synthetic training set (generated separately)

In [ ]:
# Load pre-generated training dataset and normalization stats
data = torch.load('ieee14_training_data.pt')
train_x = data['train_x']
train_y = data['train_y']
x_mean = data['x_mean']
x_std = data['x_std']

# Normalize inputs so training is numerically stable
train_x = (train_x - x_mean) / (x_std + 1e-6)


## Initialize Tensor for Physical Loss From Pandapower Model

In [ ]:
# Convert Y-bus from pandapower to PyTorch tensors for PINN usage
def get_ybus_tensors(net: pp.pandapowerNet) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Return Y-bus as real, imaginary, and complex Torch tensors for PINN usage."""
    # Access the internal admittance matrix assembled by pandapower.
    y_bus_complex = net._ppc["internal"]["Ybus"].todense()
    # Provide both split and complex forms for flexible downstream use.
    y_real = torch.tensor(np.real(y_bus_complex), dtype=torch.float32)
    y_imag = torch.tensor(np.imag(y_bus_complex), dtype=torch.float32)
    y_bus = torch.tensor(y_bus_complex, dtype=torch.complex64)
    return y_real, y_imag, y_bus

# Use function to extract admittance matrix tensor used by physics loss
_, _, Y_bus = get_ybus_tensors(net)

# Print summary of loaded data and Y-bus tensor
print(f'Loaded {train_x.shape[0]} scenarios successfully.')
print('train_x shape:', train_x.shape)
print('train_y shape:', train_y.shape)
print('Y_bus shape:', Y_bus.shape)

## Build a the Electric Power Physics Informed Neural Network (PowerPINN) 

In [ ]:


class PowerPINN(nn.Module):
    def __init__(self):
        super().__init__()
        # Build a small feedforward network
        # Input  : 28 features (P and Q measurements)
        # Hidden : two layers. 
        # Activation Function: Tanh activations to allow for both positive and negative outputs, which is important for voltage angles that can be negative.
        # Output : 28 values (14 voltages + 14 angles)
        self.net = nn.Sequential(
            nn.Linear(28, 50),
            nn.Tanh(),
            nn.Linear(50, 50),
            nn.Tanh(),
            nn.Linear(50, 28)
        )

    def forward(self, x):
        # Pass measurements through the network
        out = self.net(x)

        # Split output into [V, theta]
        # First 14 entries are voltage magnitude corrections around ~1.0 pu
        v_part = out[:, :14] + 1.0

        # Last 14 entries are phase angles (radians)
        theta_part = out[:, 14:]

        # Return one combined state vector per sample
        return torch.cat([v_part, theta_part], dim=1)

# Helper function that computes the real power mismatch given predicted voltage magnitudes/angles and the Y-bus matrix
def calculate_power_mismatch(v: torch.Tensor, theta_rad: torch.Tensor, y_bus: torch.Tensor) -> torch.Tensor:
    """Compute real-power term from predicted voltage magnitudes/angles and Y-bus."""
    # Convert polar voltage state (V, theta) into complex voltage phasors.
    v_complex = torch.complex(v * torch.cos(theta_rad), v * torch.sin(theta_rad))
    # Ohm's law in matrix form: I = YV.
    i_calc = torch.matmul(y_bus.to(v_complex.dtype), v_complex.unsqueeze(-1)).squeeze(-1)
    # Complex power S = V * conj(I); real part corresponds to active power P.
    s_calc = v_complex * torch.conj(i_calc)
    p_calc = s_calc.real
    return p_calc

# Function to compute the overall physics-based loss using the predicted state and Y-bus
def physics_loss(model, inputs, admittance_matrix, x_mean, x_std):
    # Predict state [V, theta] from input measurements
    pred = model(inputs)
    v_pred = pred[:, :14]
    theta_pred = pred[:, 14:]

    # We use the Admittance Matrix (Y_bus) here to enforce Kirchhoff's Current Law (I = YV).
    p_calc = calculate_power_mismatch(v_pred, theta_pred, admittance_matrix)
    
    # Recover the P_measured from the normalized inputs to compare against
    # The first 14 columns of inputs are Active Power (P)
    # We must un-normalize the inputs to get physical units (pu or MW/100)
    # inputs are (batch, 28). x_mean and x_std are (28,)
    # We need x_mean and x_std to be on the same device as inputs
    if x_mean.device != inputs.device:
        x_mean = x_mean.to(inputs.device)
        x_std = x_std.to(inputs.device)
        
    inputs_unnorm = inputs * (x_std + 1e-6) + x_mean
    p_measured = inputs_unnorm[:, :14] # First 14 are P
    
    # Detailed Physics Mismatch = (P_calculated - P_measured)
    mismatch = p_calc - p_measured

    # Penalize mismatch so predictions remain physically consistent
    return torch.mean(mismatch**2)


## View Training Data

In [ ]:
# Visualize training data to understand scale and structure
import matplotlib.pyplot as plt
import numpy as np

# train_x in this notebook is normalized
x_np = train_x.detach().cpu().numpy()
num_samples, num_features = x_np.shape
print(f"train_x shape: {x_np.shape} (samples x features)")
print("Feature layout: first 14 = P, last 14 = Q")

# 1) Heatmap of first N samples (normalized values)
n_show = min(120, num_samples)
plt.figure(figsize=(12, 4))
plt.imshow(x_np[:n_show], aspect='auto', cmap='coolwarm')
plt.colorbar(label='Normalized value')
plt.xlabel('Feature index (0-27)')
plt.ylabel('Sample index')
plt.title(f'Normalized Training Inputs (First {n_show} Samples)')
plt.axvline(13.5, color='black', linestyle='--', linewidth=1)
plt.text(4, -4, 'P features', fontsize=10)
plt.text(18, -4, 'Q features', fontsize=10)
plt.tight_layout()
plt.show()

# 2) Distribution summary for P and Q groups
p_vals = x_np[:, :14].reshape(-1)
q_vals = x_np[:, 14:].reshape(-1)

plt.figure(figsize=(10, 4))
plt.hist(p_vals, bins=50, alpha=0.7, label='P features (normalized)')
plt.hist(q_vals, bins=50, alpha=0.7, label='Q features (normalized)')
plt.xlabel('Normalized feature value')
plt.ylabel('Count')
plt.title('Feature Distributions: P vs Q')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 3) One scenario profile in original units (undo normalization)
sample_idx = 0
sample_norm = train_x[sample_idx]
sample_raw = (sample_norm * (x_std + 1e-6) + x_mean).detach().cpu().numpy()
p_sample = sample_raw[:14]
q_sample = sample_raw[14:]
bus = np.arange(1, 15)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(bus, p_sample)
ax[0].set_title(f'Sample {sample_idx}: Active Power P by Bus')
ax[0].set_xlabel('Bus')
ax[0].set_ylabel('P (scaled units)')
ax[0].grid(alpha=0.3)

ax[1].bar(bus, q_sample, color='tab:orange')
ax[1].set_title(f'Sample {sample_idx}: Reactive Power Q by Bus')
ax[1].set_xlabel('Bus')
ax[1].set_ylabel('Q (scaled units)')
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Training

In [ ]:
# Initialize model and optimizer
model = PowerPINN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
history = {'total': [], 'data': [], 'physics': []}

num_epochs = 2000
for epoch in range(num_epochs):
    # Reset gradients from previous step
    optimizer.zero_grad()

    # Forward pass -> predicted [V, theta]
    pred = model(train_x)

    # Data loss = prediction vs supervised labels
    loss_data = torch.mean((pred - train_y)**2)

    # Physics loss = power-flow consistency penalty
    # Updated to pass mean/std for un-normalization
    loss_phys = physics_loss(model, train_x, Y_bus, x_mean, x_std)

    # Total objective balances fit + physics
    total_loss = (10.0 * loss_data) + (0.1 * loss_phys)

    # Backprop + parameter update
    total_loss.backward()
    optimizer.step()

    # Track losses for plotting
    history['total'].append(total_loss.item())
    history['data'].append(loss_data.item())
    history['physics'].append(loss_phys.item())

    if epoch % 100 == 0:
        print(f'Epoch {epoch}: Total={total_loss.item():.6f}, Physics={loss_phys.item():.6f}')


## Plot the Results of the PowerPINN Model

In [ ]:
# Plot loss curves
plt.figure(figsize=(10, 6))
plt.plot(history['total'], label='Total Loss')
plt.plot(history['data'], label='Data Loss')
plt.plot(history['physics'], label='Physics Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Curves')
plt.legend()
plt.grid(True)
plt.show()

## Save Model for Evaluation

In [ ]:
checkpoint = {
    'model_state_dict': model.state_dict(),
    'x_mean': x_mean,
    'x_std': x_std,
    'input_dim': 28,
    'output_dim': 28
}

torch.save(checkpoint, 'pinn_model.pth')

print("Model saved successfully as pinn_model.pth")